<a href="https://colab.research.google.com/github/zou-ai-art/anima-lora-colab/blob/claude%2Fanima-lora-colab-error-b8v0f8/Anima_LoRA_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Anima-LoRA Colab

Anima 用 LoRA を Google Colab で学習するための Notebook です。モデルと tagger は同梱せず、実行時に取得します。

## 使い方

1. Colab のランタイムを GPU に変更します。
   - メニュー: `ランタイム` → `ランタイムのタイプを変更` → ハードウェアアクセラレータで `GPU` を選択
2. 上部の `すべてのセルを実行` を押します。
3. 最後の `WebUI を起動` セルまで進むと、セルの下に Anima-LoRA Colab の操作画面が表示されます。
4. 画面内のステップに沿って、画像準備、モデル確認、タグ確認、LoRA設定、作成実行を進めます。

Google Drive へ画像を直接アップロードした場合は、WebUI の Step 1 で `Drive内の画像を確認` を押してください。Colab画面から画像を選んだ場合だけ `この画面で選んだ画像をDriveへ保存` を使います。

## 1. Google Drive をマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 配布コードと依存関係を準備

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

PROJECT_DIR = Path('/content/Anima_LoRA_Colab')
HELPER_HF_REPO = 'Reo324/anima-lora-colab-helper'  # anima_lora/ と requirements-colab.txt を置いた補助コード用HFリポジトリID
HELPER_HF_REPO_TYPE = 'model'
HELPER_HF_REVISION = 'main'
SD_SCRIPTS_REPO_URL = 'https://github.com/kohya-ss/sd-scripts.git'
SD_SCRIPTS_REF = '068bcd7ffe76b2cd5012fb680a2c94e295398bbc'
# transformers は huggingface-hub 1.x で ImportError になるため、全 pip install で 1.0 未満に固定する
HF_HUB_SPEC = 'huggingface-hub>=0.34.0,<1.0'

if not HELPER_HF_REPO:
    local_project_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    if not (local_project_dir / 'requirements-colab.txt').exists():
        raise ValueError(
            'HELPER_HF_REPO が未設定です。配布前に scripts/finalize_notebook_release.py '
            'または scripts/publish_colab_release.py でHFリポジトリIDを埋め込んでください。'
        )

PIP_CONSTRAINTS = Path(tempfile.gettempdir()) / 'anima_lora_constraints.txt'
PIP_CONSTRAINTS.write_text(HF_HUB_SPEC + '\n', encoding='utf-8')


def pip_install(*args):
    command = [sys.executable, '-m', 'pip', 'install', '-c', str(PIP_CONSTRAINTS), *args]
    print('+', ' '.join(str(part) for part in command))
    subprocess.check_call(command)


def install_requirements(path: Path, *, skip_editable: bool = False, skip_packages: set[str] | None = None):
    if not path.exists():
        return
    skip_packages = {name.lower() for name in (skip_packages or set())}
    if not skip_editable and not skip_packages:
        pip_install('-r', str(path))
        return
    filtered_lines = []
    for line in path.read_text(encoding='utf-8').splitlines():
        stripped = line.strip()
        if stripped.startswith('-e ') or stripped == '.':
            print('skip Colab-unneeded editable requirement:', line)
            continue
        package_name = stripped.split('==', 1)[0].split('>=', 1)[0].split('<=', 1)[0].split('<', 1)[0].split('>', 1)[0].split('~=', 1)[0].split('[', 1)[0].strip().lower()
        if package_name in skip_packages:
            print('skip Colab-managed binary requirement:', line)
            continue
        filtered_lines.append(line)
    with tempfile.NamedTemporaryFile('w', suffix='.txt', delete=False, encoding='utf-8') as handle:
        handle.write('\n'.join(filtered_lines) + '\n')
        filtered_path = Path(handle.name)
    pip_install('-r', str(filtered_path))


pip_install(HF_HUB_SPEC)

try:
    from google.colab import userdata
    BOOTSTRAP_HF_TOKEN = userdata.get('HF_TOKEN') or None
except Exception:
    BOOTSTRAP_HF_TOKEN = None

if HELPER_HF_REPO:
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=HELPER_HF_REPO,
        repo_type=HELPER_HF_REPO_TYPE,
        revision=HELPER_HF_REVISION,
        local_dir=str(PROJECT_DIR),
        allow_patterns=['anima_lora/**', 'requirements-colab.txt'],
        token=BOOTSTRAP_HF_TOKEN,
    )

if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if not (PROJECT_DIR / 'requirements-colab.txt').exists():
    raise RuntimeError('補助コードが見つかりません。HELPER_HF_REPO を設定してください。')

sys.path.insert(0, str(PROJECT_DIR))
print('PROJECT_DIR =', PROJECT_DIR)

requirements = PROJECT_DIR / 'requirements-colab.txt'
install_requirements(requirements, skip_packages={'numpy'})

sd_scripts_dir = Path('/content/sd-scripts')
if not sd_scripts_dir.exists():
    subprocess.check_call(['git', 'clone', SD_SCRIPTS_REPO_URL, str(sd_scripts_dir)])
    subprocess.check_call(['git', '-C', str(sd_scripts_dir), 'checkout', SD_SCRIPTS_REF])

sd_requirements = sd_scripts_dir / 'requirements.txt'
install_requirements(sd_requirements, skip_editable=True, skip_packages={'numpy'})

## 3. Colab環境を確認

In [ ]:
from anima_lora.colab.setup import check_colab_environment, format_preflight_report

env_report = check_colab_environment(require_gpu=True)
print(format_preflight_report(env_report))
if not env_report.ok:
    raise RuntimeError(format_preflight_report(env_report))

## 4. 作業ディレクトリを作成

モデルキャッシュと学習結果は `/content/drive/MyDrive/_Anima_LoRA_Colab_Workspace` 配下に保存します。Google Drive側に残るため、同じファイルがあれば次回は再利用します。

In [ ]:
from anima_lora.colab.setup import ColabPaths, ensure_work_dirs

paths = ensure_work_dirs(ColabPaths())
print('Drive cache root:', paths.cache_root)
print('Anima models:', paths.models_root)
print('Tagger assets:', paths.tagger_dir)
print('Output:', paths.output_dir)
paths

## 5. モデルと tagger を取得

In [ ]:
from anima_lora.colab.downloads import download_anima_models, download_tagger_assets
from anima_lora.colab.secrets import read_secret

HF_TOKEN = read_secret('HF_TOKEN') or None
TAGGER_HF_REPO = 'Reo324/anima-lora-tagger'  # tagger用HFリポジトリID
EMBEDDED_TAGGER_LABEL_KEY = ''
if not TAGGER_HF_REPO:
    raise ValueError(
        'TAGGER_HF_REPO が未設定です。tagger用HFリポジトリIDを、'
        '配布前に scripts/finalize_notebook_release.py または scripts/publish_colab_release.py で埋め込んでください。'
    )

download_anima_models(paths.models_root, token=HF_TOKEN)
download_tagger_assets(TAGGER_HF_REPO, paths.tagger_dir, token=HF_TOKEN)

## 6. WebUI を起動

In [ ]:
from anima_lora.colab.webui import launch_webui

launch_webui(paths=paths, sd_scripts_dir=sd_scripts_dir)